<a href="https://colab.research.google.com/github/gigimcc4/IBM_for_researchers/blob/main/Copy_of_Tutorial_4_Agent_Workflows_IBMBEE(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **IMPORTANT — FIRST — MAKE A COPY OF THIS NOTEBOOK, OR NOTHING WILL BE SAVED!**
## **FILE → SAVE A COPY IN DRIVE / GITHUB**

# Day 4 Extension · Agent-Assisted Workflows for Your Project
## Building Your Track-Specific Agentic Tool

---
*Created for the Nagoya University 5-Day Faculty Seminar on Teaching with AI*  
**Dr. Jeanne McClure, PhD** | NC State University Data Science Academy / Ars Innovate

---

### 📜 License

© 2026 Dr. Jeanne McClure. This work is licensed under a [Creative Commons Attribution-NonCommercial 4.0 International License (CC BY-NC 4.0)](https://creativecommons.org/licenses/by-nc/4.0/).

This material was in part developed with support from the National Science Foundation, DUE-2313644. All opinions, findings, conclusions and recommendations expressed herein are those of the authors and do not necessarily reflect the views of the Foundation.

---

| | |
|---|---|
| **Time** | 60 minutes |
| **Prerequisite** | Day 4 Main Session (Agentic AI) |
| **What you build** | An agent workflow tailored to your Choose Your Adventure track |

### Your Track (from Day 1)

| Track | Today's Focus |
|-------|---------------|
| **Track A - For My Students** | Build an agent-assisted assignment with audit checklist |
| **Track D - For My Teaching** | Build an agent-assisted course material generator |
| **Track B - For My Research** | Build an agent that assists with your research workflow |
| **Track C - For My Department** | Design an agent-based assessment template for adoption |

---

© 2026 Dr. Jeanne McClure · CC BY-NC 4.0 · Supported by NSF DUE-2313644


---
## ⚠️ Before You Run This Notebook — Read This First

This notebook uses **IBM BeeAI** — a true agentic framework where the model drives its own loop.
BeeAI connects to **Ollama**, which runs the IBM Granite model locally on your machine.

**This is different from the Day 4 main session demo**, which ran Granite directly in Colab via Hugging Face.
BeeAI requires Ollama to be running — and Ollama must be installed on a local machine or run fresh in Colab each session.

---

### Choose your path

| Where are you running this? | What to do |
|-----------------------------|------------|
| **Your own laptop / desktop** | Follow the Local Setup section below — do this once, works every time |
| **Google Colab** | Skip to the Colab Setup section — Ollama reinstalls each session (~3 min) |

In [ ]:
import subprocess, time, requests

# Install zstd dependency first
subprocess.run('apt-get install -y zstd', shell=True)

# Now install Ollama
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True)

# Start the server
subprocess.Popen('ollama serve', shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(6)

# Pull the model
subprocess.run('ollama pull granite3.3:2b', shell=True)

# Confirm
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f'✓ Ready — models: {models}')
except:
    print('⚠️ Not responding yet — wait 10 seconds and run the confirm line again')

✓ Ready — models: ['granite3.3:2b']


---
## ☁️ COLAB SETUP — Run this notebook in Google Colab

Colab resets its virtual machine when the session ends.
That means Ollama must be reinstalled each session — takes about 3 minutes.

> **Note:** This is why we recommend running locally if you plan to use this notebook regularly.
> For a one-time workshop demo, Colab works fine.

**Before running any cells:**
Go to **Runtime → Change runtime type → T4 GPU → Save**

Then run the setup cell below — it installs Ollama, starts the server, pulls Granite, and confirms everything is ready.
Wait for **Done: Ready** before running any other cells.

### Colab Setup Cell

Run the cell below. It installs Ollama, starts the server, downloads IBM Granite, and confirms everything is ready.
Wait for **Done: Ready** before moving on.


In [ ]:
# Setup (skip if already done in main session)
import subprocess, time, os, requests, json

# Check if Ollama is running
try:
    r = requests.get('http://localhost:11434/api/tags')
    models = [m['name'] for m in r.json().get('models', [])]
    if any('granite' in m for m in models):
        print('Done: Ollama + Granite already running from main session!')
    else:
        raise Exception('Granite not found')
except:
    print('Setting up from scratch...')
    !curl -fsSL https://ollama.com/install.sh | sh 2>/dev/null
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    !ollama pull granite3.3:2b
    print('Done: Setup complete!')

!pip install -q beeai-framework requests 2>/dev/null
print('Done: Ready to build agent workflows!')


Done: Ollama + Granite already running from main session!
Done: Ready to build agent workflows!


In [ ]:
# ── INSTALL PYTHON PACKAGES ───────────────────────────────────────────────────
subprocess.run('pip install -q "beeai-framework[wikipedia]" requests pandas', shell=True)

# ── CONFIRM ───────────────────────────────────────────────────────────────────
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f'✓ Ready — models available: {models}')
except:
    print('⚠️ Ollama not responding.')
    print('   Colab: wait 10 seconds and run this cell again')
    print('   Mac: open Ollama from Applications and run this cell again')
    print('   Windows: open Ollama from Start menu and run this cell again')

✓ Ready — models available: ['granite3.3:2b']


In [ ]:
# Shared helper: talk to Granite
import requests

def ask_granite(prompt, system="You are a helpful assistant."):
    response = requests.post('http://localhost:11434/api/chat', json={
        'model': 'granite3.3:2b',
        'messages': [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': prompt}
        ],
        'stream': False
    })
    return response.json()['message']['content']

from beeai_framework.agents.requirement import RequirementAgent
from beeai_framework.tools.search.wikipedia import WikipediaTool
from beeai_framework.tools.think import ThinkTool
from beeai_framework.adapters.ollama.backend.chat import OllamaChatModel

llm = OllamaChatModel(model_id='granite3.3:2b', url='http://127.0.0.1:11434')
print('Done: LLM and agent framework loaded.')


Done: LLM and agent framework loaded.


---
# Track A: Agent-Assisted Assignment Builder

*Skip to Track B or C if those match your path.*

### Your Goal
Design an assignment where students use an AI agent to complete a task,
then audit the agent's work using the checklist from the main session.

### The Recipe

```
1. Define the task (what students ask the agent to do)
2. Choose the agent's tools (what it can access)
3. Run the agent yourself (test it)
4. Design the audit (what students evaluate)
5. Build the rubric (how you assess the human contribution)
```


### Step 1: Define Your Assignment Task

**Edit the cell below** to describe a task from your course that students
could do with an AI agent's help.

Good tasks have:
- A clear goal
- Multiple steps (so the agent needs to plan)
- Room for human judgment (so students add value)


In [ ]:
# 🎓 TRACK A - Step 1: Define your assignment task
#  Edit these variables for your course!

my_course = "Introduction to Biology"  # ← Your course
my_task = (
    "Research the impact of climate change on coral reef ecosystems. "
    "Summarize the key findings and suggest three research questions "
    "that a student could investigate further."
)  # ← Your assignment task

# What tools should the agent have?
agent_tools = "Wikipedia search + reasoning"  # ← Describe the tools

print(f' Course: {my_course}')
print(f' Task: {my_task}')
print(f' Agent tools: {agent_tools}')
print(f'\n Next: Run the agent on this task and see what it produces.')


 Course: Introduction to Biology
 Task: Research the impact of climate change on coral reef ecosystems. Summarize the key findings and suggest three research questions that a student could investigate further.
 Agent tools: Wikipedia search + reasoning

 Next: Run the agent on this task and see what it produces.


In [ ]:
# 🎓 TRACK A - Step 2: Run the agent on your task

student_agent = RequirementAgent(
    llm=llm,
    tools=[WikipediaTool(), ThinkTool()],
)

async def test_assignment():
    print(f' Running agent on your assignment task...')
    print(f'   Task: "{my_task[:80]}..."')
    print('=' * 60)

    result = await student_agent.run(my_task)

    print(f'\nDone: AGENT OUTPUT:')
    print(result.state.answer.text[:800])

    print('\n' + '=' * 60)
    print(' NOW DESIGN THE AUDIT:')
    print('   1. What did the agent get right?')
    print('   2. What did it miss or get wrong?')
    print('   3. What would a student need to ADD to make this excellent?')
    print('   4. How would you SCORE the human contribution?')

await test_assignment()


 Running agent on your assignment task...
   Task: "Research the impact of climate change on coral reef ecosystems. Summarize the ke..."

Done: AGENT OUTPUT:
I couldn't find specific Wikipedia articles directly addressing the impact of climate change on coral reef ecosystems. However, based on extensive research and reports from scientific bodies like the Intergovernmental Panel on Climate Change (IPCC), I can summarize the key findings:

 NOW DESIGN THE AUDIT:
   1. What did the agent get right?
   2. What did it miss or get wrong?
   3. What would a student need to ADD to make this excellent?
   4. How would you SCORE the human contribution?


### Step 3: Design Your Complete Assignment Package

**Double-click to edit this template:**

---

**Assignment Title:** *[Your title]*

**Instructions to Students:**

> Use the AI research agent to investigate [topic]. The agent can search
> Wikipedia and reason about its findings. Your job is NOT just to submit
> what the agent produces. Your job is to:
>
> 1. Run the agent on the task described below
> 2. Evaluate the agent's output using the audit checklist
> 3. Identify at least 2 errors, gaps, or limitations
> 4. Write your own analysis that builds on (and improves) the agent's work
>
> **You submit:** The agent's full output + your audit + your revised analysis

**Rubric:**

| Criterion (25% each) | Excellent (A) | Adequate (B-C) | Insufficient (D-F) |
|---|---|---|---|
| Agent Direction | Clear, specific task that produces useful output | Vague task producing generic output | Copy-pasted task with no customization |
| Audit Quality | Identifies subtle errors; checks facts independently | Identifies obvious errors | Accepts output without evaluation |
| Human Contribution | Substantial original analysis beyond agent's work | Some additions but largely derivative | Minimal change from agent's output |
| Integration | Seamlessly combines AI findings with own expertise | Awkward separation between AI and human parts | AI output presented as own work |


---
# Track D: Course Material Generator Agent

*Skip to Track B or C if those match your path.*

### Your Goal
Build an agent workflow that helps you **create, revise, or redesign course
materials** - using AI as a productivity partner that searches for relevant
content, drafts material, and structures it to your specifications.

### Where Agents Help with Teaching Materials

| Material Type | Agent Can... | You Must... |
|---------------|-------------|-------------|
| **Lesson plans** | Research best practices, draft activity sequences | Ensure alignment with YOUR objectives and students |
| **Slide decks** | Generate content outlines, suggest visuals | Refine for your voice, check accuracy |
| **Rubrics** | Draft criteria from learning objectives | Calibrate levels to your standards |
| **Activities** | Generate variations, adapt for different levels | Test feasibility, align with pedagogy |
| **Assessments** | Draft questions, create answer keys | Verify accuracy, ensure fair coverage |


In [ ]:
# 📝 TRACK D - Build a course material generator agent
#  Edit the material_context for your course!

material_context = {
    "course": "Introduction to Statistics",  # ← Your course
    "material_type": "lesson plan with active learning activities",  # ← What you're building
    "topic": "Central Limit Theorem",  # ← The specific topic
    "students": "second-year undergraduates, mixed STEM majors",  # ← Your students
    "session_length": "75 minutes",  # ← How long
}

material_agent = RequirementAgent(
    llm=llm,
    tools=[WikipediaTool(), ThinkTool()],
)

async def material_builder():
    prompt = (
        f"You are a curriculum design assistant for a {material_context['course']} course. "
        f"Create a {material_context['material_type']} on {material_context['topic']}. "
        f"Students: {material_context['students']}. "
        f"Session length: {material_context['session_length']}. "
        f"Search for relevant pedagogical approaches and content, then think carefully "
        f"about how to structure an engaging session. Include timing, activities, "
        f"discussion prompts, and formative assessment checks."
    )

    print(f'📝 Material Agent building: {material_context["material_type"]}')
    print(f'   Topic: {material_context["topic"]}')
    print(f'   Course: {material_context["course"]}')
    print('=' * 60)

    result = await material_agent.run(prompt)
    print(f'\nDone: MATERIAL AGENT OUTPUT:')
    print(result.state.answer.text[:800])

    print('\n' + '=' * 60)
    print(' YOUR QUALITY CHECK BEFORE DEPLOYING:')
    print('   1. Is the content accurate for my discipline?')
    print('   2. Is the difficulty appropriate for my students?')
    print('   3. Does this reflect my teaching style and voice?')
    print('   4. Is this practical (time, resources, class size)?')
    print('   5. What do I need to add, remove, or revise?')

await material_builder()


📝 Material Agent building: lesson plan with active learning activities
   Topic: Central Limit Theorem
   Course: Introduction to Statistics

Done: MATERIAL AGENT OUTPUT:
### Lesson Plan: Introduction to Statistics - Central Limit Theorem

**Objective:**
By the end of this session, students will understand the Central Limit Theorem (CLT), its significance, and be able to apply it to real-world scenarios.

**Target Audience:** Second-year undergraduate students from mixed STEM majors.
**Session Duration:** 75 minutes

**Lesson Structure:**

1. **Introduction (10 minutes)**
   - Briefly review the concept of probability distributions (5 mins)
   - Introduce the Central Limit Theorem (CLT) as a powerful tool for understanding the behavior of sample means (5 mins)

2. **Interactive Lecture (15 minutes)**
   - Explain the CLT mathematically and conceptually
   - Use visual aids (graphs, histograms) to illustrate the normal distribution of sample means
   - Discu

 YOUR QUALITY CHECK BEFORE 

### 📝 Teaching Materials Template - Double-click to edit

**My course:** *(name and level)*

**Material I built today:** *(type and topic)*

**Agent workflow for creating materials:**
1. Step: *(what I tell the agent)* → My quality check: *(how I verify)*
2. Step: *(refinement prompt)* → My quality check: *(what I adjust)*
3. Step: *(formatting/finalizing)* → My quality check: *(final review)*

**What the agent CANNOT replace:**
- *(my knowledge of my specific students)*
- *(my pedagogical judgment)*
- *(my teaching voice and style)*

**Time savings estimate:**
- Manual creation: *(hours)* → Agent-assisted: *(hours)*

**How I'll use this for Day 5:**
- *(describe your capstone deliverable - a complete course material package)*


---
# Track B: Research Workflow Agent

*Skip to Track C if that matches your path.*

### Your Goal
Build an agent workflow that assists with a specific step in your research
process - literature search, data analysis interpretation, methodology review,
or manuscript preparation.

### Research Workflow Stages Where Agents Help

| Stage | Agent Can... | Human Must... |
|-------|-------------|---------------|
| **Literature search** | Search databases, summarize papers | Evaluate relevance, identify gaps |
| **Data interpretation** | Run statistical analyses, generate visualizations | Interpret meaning, connect to theory |
| **Methodology review** | Compare methods across papers, identify patterns | Judge appropriateness for your context |
| **Writing assistance** | Draft sections, suggest structure | Ensure accuracy, maintain voice, add insight |


In [ ]:
# 🔬 TRACK B - Build a research assistant agent
#  Edit the research_context for your work!

research_context = {
    "field": "Educational Technology",  # ← Your field
    "topic": "AI-assisted assessment in higher education",  # ← Your topic
    "stage": "literature review",  # ← Which stage you want help with
    "specific_question": (
        "What frameworks exist for designing AI-partnered assessments "
        "that maintain academic integrity while promoting AI literacy?"
    ),  # ← Your specific research question
}

research_agent = RequirementAgent(
    llm=llm,
    tools=[WikipediaTool(), ThinkTool()],
)

async def research_assistant():
    prompt = (
        f"You are a research assistant in {research_context['field']}. "
        f"Help with the {research_context['stage']} stage. "
        f"Research question: {research_context['specific_question']} "
        f"Search for relevant background information, then think carefully "
        f"about what the key themes, debates, and gaps are."
    )

    print(f'🔬 Research Agent working on: {research_context["stage"]}')
    print(f'   Question: {research_context["specific_question"][:80]}...')
    print('=' * 60)

    result = await research_agent.run(prompt)
    print(f'\nDone: RESEARCH AGENT OUTPUT:')
    print(result.state.answer.text[:800])

    print('\n' + '=' * 60)
    print(' YOUR AUDIT AS A RESEARCHER:')
    print('   1. Are the sources/claims the agent found credible?')
    print('   2. What important work did the agent miss?')
    print('   3. Does the agent\'s synthesis match your domain expertise?')
    print('   4. What would you add or correct before using this in a paper?')

await research_assistant()


🔬 Research Agent working on: literature review
   Question: What frameworks exist for designing AI-partnered assessments that maintain acade...

Done: RESEARCH AGENT OUTPUT:
The initial search through Wikipedia did not yield direct information on AI-partnered assessment frameworks. To address this research question, I will explore academic databases such as Google Scholar, JSTOR, and ERIC for relevant frameworks. I will also consider conference proceedings and research papers from reputable institutions. I will look for frameworks that focus on maintaining academic integrity and promoting AI literacy.

 YOUR AUDIT AS A RESEARCHER:
   1. Are the sources/claims the agent found credible?
   2. What important work did the agent miss?
   3. Does the agent's synthesis match your domain expertise?
   4. What would you add or correct before using this in a paper?


### 🔬 Research Workflow Template - Double-click to edit

**My research area:** *(field and topic)*

**Workflow I want to agent-assist:**
1. Step: *(what the agent does)* → My audit: *(how I verify)*
2. Step: *(what the agent does)* → My audit: *(how I verify)*
3. Step: *(what the agent does)* → My audit: *(how I verify)*

**What the agent CANNOT replace in my workflow:**
- *(list the irreducibly human parts)*

**How I'll use this for Day 5:**
- *(describe your capstone deliverable)*


---
# Track C: Department Assessment Template

### Your Goal
Design a department-level assessment template that other faculty can adopt.
This becomes a shared resource - a **policy-aligned, practical tool** that
helps colleagues integrate AI into assessment thoughtfully.

### What a Department Template Needs

| Component | Purpose |
|-----------|--------|
| **AI Use Policy Statement** | Clear language for syllabi about when and how AI can be used |
| **Assessment Design Checklist** | The 4-question framework adapted for your discipline |
| **Student-Facing Audit Template** | A reusable checklist students submit with AI-assisted work |
| **Rubric Modification Guide** | How to add "human contribution" criteria to existing rubrics |
| **Integrity vs. Competency Decision Tree** | When to prohibit, make transparent, or partner with AI |


In [ ]:
# 🏛️ TRACK C - Generate a draft AI policy statement
# The agent will draft; you'll revise for your department's context.

department = "Biology"  # ← Your department
institution_type = "research university"  # ← Your institution type

policy_prompt = f"""
Draft an AI use policy statement for a {department} department at a
{institution_type}. The policy should:

1. Acknowledge that AI tools are part of professional practice
2. Distinguish between prohibited, transparent, and partnered AI use
3. Require documentation of AI assistance
4. Focus on competency demonstration rather than AI detection
5. Be concise enough to fit in a syllabus (under 200 words)

Write the policy statement now.
"""

print(f'🏛️ Generating draft AI policy for {department}...')
print('=' * 60)
policy = ask_granite(policy_prompt,
    system="You are an educational policy writer. Write clear, practical policy language.")
print(policy)

print('\n' + '=' * 60)
print(' YOUR AUDIT:')
print('   1. Is this appropriate for your department\'s culture?')
print('   2. What would your colleagues push back on?')
print('   3. What\'s missing for your specific disciplinary context?')
print('   4. How would you modify this before proposing it?')


🏛️ Generating draft AI policy for Biology...
**AI Use Policy for Biology Department**

"In our commitment to academic integrity and responsible innovation, the Biology Department acknowledges that Artificial Intelligence (AI) tools serve as an integral part of professional practice. We distinguish between three categories of AI utilization:

1. **Prohibited Use**: Direct use of AI for grading or marking assignments, excluding cases where students have explicit consent to use AI for self-check (with explicit disclosure).

2. **Transparent Use**: Faculty and students may employ AI as a support tool for research, analysis, or learning, provided full transparency to peers and supervisors about AI's role in the process.

3. **Partnered Use**: Collaborations with AI developers to co-create innovative solutions, adhering to ethical standards and respecting intellectual property rights.

All instances of AI utilization necessitate meticulous documentation, outlining the AI's role, the purpose 

### 🏛️ Department Template - Double-click to edit

**Department:** *(name)*

**AI Use Policy Statement (revised from agent draft):**
> *(paste your revised version here)*

**Assessment Design Checklist for Faculty:**
- [ ] Identify what AI can do on this assignment
- [ ] Define the human competency being assessed
- [ ] Design how student thinking becomes visible
- [ ] Create rubric criteria for the human contribution
- [ ] Decide: AI-Prohibited / AI-Transparent / AI-Partnered

**Student Submission Template:**
- [ ] AI agent output (full transcript)
- [ ] Student audit checklist (completed)
- [ ] Student's revised/extended analysis
- [ ] Reflection: What did you learn that AI couldn't tell you?

**Implementation Plan:**
- Pilot in: *(which course, when)*
- Share with department: *(when, how)*
- Collect feedback: *(from whom, when)*


---
## What to Bring to Day 5 (Tomorrow)

**All tracks:**
- Your completed template from above
- The agent output + your audit (to demonstrate your process)
- One thing that surprised you about how the agent performed

**Student Track (A):** Your assignment package (instructions, checklist, rubric)

**Teaching Track (D):** Your agent-assisted material generator workflow + sample output

**Research Track (B):** Your agent-assisted workflow with audit protocol

**Department Track (C):** Your department template package (policy + checklist + rubric guide)

---

**Tomorrow's structure:**
1. **Build time** (45 min) - Finalize your product using all the tools from the week
2. **Gallery walk** (30 min) - Share your product, get cross-disciplinary feedback
3. **Commitment** (15 min) - Write your implementation plan for next semester

---
*© 2026 Dr. Jeanne McClure · CC BY-NC 4.0 · NSF DUE-2313644*


---
## 💻 LOCAL SETUP — How to run this notebook on your own computer

Do this once. After setup everything persists — no reinstall needed next time.

---

### Step 1 · Install Ollama

**Mac:**
1. Open the **Terminal** app
   - Press **Command + Space**, type `Terminal`, press Enter
2. Copy and paste this line, then press Enter:
```
curl -fsSL https://ollama.com/install.sh | sh
```
3. Wait for it to finish — you will see `Install complete`

**Windows:**
1. Open a browser and go to: [https://ollama.com/download](https://ollama.com/download)
2. Click **Download for Windows**
3. Open the downloaded file and click through the installer

**Linux:**
1. Open Terminal
2. Run: `curl -fsSL https://ollama.com/install.sh | sh`

---

### Step 2 · Download the IBM Granite Model

This downloads the AI model to your computer. It is about 1.5 GB and only happens once.

**Mac:** In Terminal, run:
```
ollama pull granite3.3:2b
```

**Windows:** Open **Command Prompt**
- Press the **Windows key**, type `cmd`, press Enter
- Then run:
```
ollama pull granite3.3:2b
```

Wait for the progress bar to complete. After this the model lives on your computer permanently.

---

### Step 3 · Start Ollama (do this each time before opening the notebook)

**Mac:**
1. Open **Finder**
2. Click **Applications** in the left sidebar
3. Find **Ollama** and double-click it
4. Nothing dramatic happens — no window opens
5. Look at the very top right of your screen (the row of small icons near the clock)
6. You will see a small llama icon 🦙 — that means Ollama is running

**Windows:**
1. Click the **Start** button (Windows logo, bottom left)
2. Type `Ollama` and press Enter
3. Nothing dramatic happens — no window opens
4. Look at the bottom right of your screen near the clock
5. You may need to click the small **^** arrow to see hidden icons
6. You will see an Ollama icon there — that means it is running

**Linux:**
1. Open Terminal and run: `ollama serve`
2. Leave that Terminal window open while you work

**To confirm Ollama is running on any system:**
Open any browser and go to: `http://localhost:11434`
You should see the words: **Ollama is running**
If you see that, you are ready.

---

### Step 4 · Install Jupyter (if you do not have it already)

**Mac:** In Terminal, run:
```
pip install jupyter notebook
```

**Windows:** In Command Prompt, run:
```
pip install jupyter notebook
```

If you see an error saying `pip` is not found, try `pip3` instead.

---

### Step 5 · Install the Python packages this notebook needs

In Terminal (Mac) or Command Prompt (Windows), run:
```
pip install beeai-framework requests pandas
```

---

### Step 6 · Open the notebook

1. Download this `.ipynb` file to your computer — save it somewhere you can find it, like your Desktop or Downloads folder
2. Open Terminal (Mac) or Command Prompt (Windows)
3. Navigate to where you saved the file. For example if it is in Downloads:
```
cd Downloads
```
4. Start Jupyter:
```
jupyter notebook
```
5. A browser window will open automatically showing your files
6. Click the notebook file to open it
7. Run each cell top to bottom using **Shift + Enter**

---

> ✓ **Next time:** Start Ollama (Step 3) → open Terminal → run `jupyter notebook` → open the file.
> Everything else is already installed.